# Лабораторная работа №4 — вероятность

Задания по событиям, биномиальной модели, имитации Монте-Карло и байесовскому
обновлению. В заключительном разделе используется локальный набор Bike Sharing.
Все случайные процедуры воспроизводятся при `RANDOM_STATE = 42`.
При локальном запуске из каталога `lab4` интернет не требуется.
В Google Colab нужный CSV автоматически загружается из этого репозитория.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

RANDOM_STATE = 42
DATA_DIR = Path("data/bike-sharing/raw")
if not (DATA_DIR / "hour.csv").is_file():
    try:
        import google.colab
    except ImportError as exc:
        raise FileNotFoundError(
            "Не найден data/bike-sharing/raw/hour.csv. Запустите Jupyter из каталога lab4."
        ) from exc
    from hashlib import sha256
    from urllib.request import urlopen
    url = ("https://raw.githubusercontent.com/huksleva/DataScience/main/"
           "lab4/data/bike-sharing/raw/hour.csv")
    try:
        with urlopen(url, timeout=60) as response:
            csv_bytes = response.read()
    except OSError as exc:
        raise RuntimeError(f"Не удалось загрузить данные Bike Sharing из {url}") from exc
    expected_sha256 = "e03de4ee4ef4dc376ac6e04bf829673c6269e8eba5c60fa121640fa2f829504f"
    if sha256(csv_bytes).hexdigest() != expected_sha256:
        raise ValueError("Контрольная сумма hour.csv не совпадает с manifest.json")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "hour.csv").write_bytes(csv_bytes)
    print("Colab: hour.csv автоматически загружен и проверен.")

plt.rcParams.update({"figure.dpi": 115, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 1. События дождя и доставки

**Обозначения и модель.** $\bar A$ — дополнение события $A$;
$A\cap B$ — совместное наступление; $A\cup B$ — наступление хотя
бы одного события. При $P(B)>0$ верно

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}.
$$

Независимость означает $P(A\cap B)=P(A)P(B)$. Пусть
$P(R)=0.30$, $P(D)=0.40$; в задании 1.1 независимость
предполагается, а в задании 1.3 заменяется условием
$P(D\mid R)=0.20$.


### Ответ: события дождя и доставки

Пусть $R$ — дождь, $D$ — доставка зонта. При независимости
$P(R\cap D)=P(R)P(D)=0.30\cdot0.40=\mathbf{0.12}$.
Событие $\bar R\cup D$ дополняет $R\cap\bar D$, поэтому
$P(\bar R\cup D)=1-0.30(1-0.40)=\mathbf{0.82}$.
Формула объединения даёт то же число:
$P(\bar R)+P(D)-P(\bar R\cap D)=0.70+0.40-0.28=0.82$.

Если вместо независимости известно $P(D\mid R)=0.20$, то
$P(R\cap D)=P(R)P(D\mid R)=0.30\cdot0.20=\mathbf{0.06}$.
Это другая модель зависимости; маргинальных $P(R)$ и $P(D)$ без
дополнительного условия для совместной вероятности недостаточно.
Числа описывают заданную модель, а не реальные частоты дождя или доставки.


In [ ]:
p_rain, p_delivery = 0.30, 0.40
p_joint_independent = p_rain * p_delivery
p_not_rain_or_delivery = 1 - p_rain * (1 - p_delivery)
p_joint_conditional = p_rain * 0.20

assert np.isclose(p_joint_independent, 0.12)
assert np.isclose(p_not_rain_or_delivery, 0.82)
assert np.isclose(p_joint_conditional, 0.06)
display(pd.Series({
    "P(R ∩ D), независимость": p_joint_independent,
    "P(не R ∪ D), независимость": p_not_rain_or_delivery,
    "P(R ∩ D), P(D | R) = 0.20": p_joint_conditional,
}, name="вероятность").to_frame())


## 2. Биномиальная модель неявок

**Обозначения и модель.** Для $X\sim\operatorname{Binom}(n,p)$
величина $X$ считает успехи в $n$ независимых испытаниях с общим
$p$. Функция `binom.sf(k - 1, n, p)` возвращает $P(X\ge k)$.
При имитации $I_i=\mathbf1\{X_i\ge50\}$, а

$$
\widehat p_M=\frac1M\sum_{i=1}^M I_i
$$

оценивает искомую вероятность. В продолжении вместимость самолёта
$C=100$, а риск отказа равен

$$
P(b-X>C)=P(X\le b-C-1).
$$


### Ответ: биномиальная модель, имитация и перепродажа мест

Пусть $X$ — число неявок из $n=137$ бронирований. При независимых
одинаковых вероятностях неявки $p=0.40$ имеем $X\sim\mathrm{Binom}(137,0.40)$.
Точный расчёт $P(X\ge50)=\operatorname{sf}(49)=\mathbf{0.822096}$.
Доля событий в 100 000 имитаций с зерном 42 равна $\mathbf{0.822670}$;
разница $0.000574$ укладывается в случайную погрешность Монте-Карло.

Для $b$ бронирований вместимость $C=100$. Отказ возможен при
$b-X>C$, то есть $X\le b-C-1$, где теперь $X\sim\mathrm{Binom}(b,0.40)$.
По всем $b=100,\ldots,160$ максимум при риске не более 5% —
$\mathbf{b^*=151}$: риск $\mathbf{0.048869}$, а при $b^*+1=152$ уже
$\mathbf{0.060715}$. Модель предполагает независимые неявки с одним
неизменным $p$ и не учитывает общие причины неявок, изменение поведения
пассажиров и последствия отказа в посадке. Имитация проверяет расчёт,
но не подтверждает эти предположения в реальной авиакомпании.


In [ ]:
n, p_no_show = 137, 0.40
p_at_least_50 = stats.binom.sf(49, n, p_no_show)
draws = np.random.default_rng(RANDOM_STATE).binomial(n, p_no_show, size=100_000)
p_simulated = np.mean(draws >= 50)
mc_se = np.sqrt(p_simulated * (1 - p_simulated) / len(draws))
print(f"P(X ≥ 50), точно: {p_at_least_50:.6f}")
print(f"P(X ≥ 50), Монте-Карло: {p_simulated:.6f}; MC SE ≈ {mc_se:.6f}")

capacity = 100
bookings = np.arange(100, 161)
risks = stats.binom.cdf(bookings - capacity - 1, bookings, p_no_show)
acceptable = bookings[risks <= 0.05]
max_b = int(acceptable.max())
risk_at_max = float(risks[bookings == max_b][0])
risk_next = float(risks[bookings == max_b + 1][0])
assert risk_at_max <= 0.05 < risk_next
assert np.all(risks[bookings > max_b] > 0.05)
print(f"b* = {max_b}: риск {risk_at_max:.6f}")
print(f"b* + 1 = {max_b + 1}: риск {risk_next:.6f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bookings, risks, marker=".", markersize=4)
ax.axhline(0.05, color="tab:red", ls="--", label="Порог 5%")
ax.axvline(max_b, color="tab:green", ls=":", label=f"Максимум b* = {max_b}")
ax.set(xlabel="Число бронирований b", ylabel="P(отказ в посадке)",
       title="Риск перепродажи мест при C = 100")
ax.legend()
plt.show()


## 3. Честная монета и Beta-обновление

В 19 бросках наблюдались 15 орлов. При честной монете найдите
$P(X\le4)+P(X\ge15)$. Затем используйте априорное распределение
(*prior*) $\operatorname{Beta}(\alpha,\beta)$. После $s$ орлов и
$f$ решек апостериорное распределение (*posterior*) равно

$$
\operatorname{Beta}(\alpha+s,\beta+f),\qquad
\operatorname{E}[p\mid\text{данные}]=\frac{\alpha+s}{\alpha+\beta+s+f}.
$$

Центральный байесовский достоверный интервал (*credible interval*)
строится по квантилям posterior и зависит от prior; он не является
частотным доверительным интервалом.


### Ответ: монета и байесовское обновление

При честной независимой монете $X\sim\mathrm{Binom}(19,0.5)$.
Симметричная двусторонняя хвостовая вероятность
$P(X\le4)+P(X\ge15)=\mathbf{0.019211}$.
Она мала для данной нулевой модели, но сама по себе не доказывает
физическую нечестность монеты.

Для вероятности орла $p$ задан prior $\mathrm{Beta}(1,1)$.
После $s=15$ орлов и $f=4$ решек posterior равен
$\mathrm{Beta}(1+s,1+f)=\mathrm{Beta}(16,5)$.
Его среднее $16/21=\mathbf{0.761905}$, центральный 95%-й
байесовский достоверный интервал — $\mathbf{[0.563386;\ 0.913429]}$.
При заданных данных и prior posterior приписывает этому интервалу
вероятность 95%; вывод зависит от prior и модели независимых бросков.
Это не частотный доверительный интервал.


In [ ]:
p_two_sided = stats.binom.cdf(4, 19, 0.5) + stats.binom.sf(14, 19, 0.5)
alpha_prior, beta_prior = 1, 1
successes, failures = 15, 4
alpha_post = alpha_prior + successes
beta_post = beta_prior + failures
posterior_mean = alpha_post / (alpha_post + beta_post)
posterior_interval = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
print(f"Двусторонняя вероятность: {p_two_sided:.6f}")
print(f"Posterior: Beta({alpha_post}, {beta_post}), среднее {posterior_mean:.6f}")
print(f"95%-й credible interval: [{posterior_interval[0]:.6f}; {posterior_interval[1]:.6f}]")

grid = np.linspace(0, 1, 501)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, stats.beta.pdf(grid, alpha_prior, beta_prior), label="Prior Beta(1, 1)")
ax.plot(grid, stats.beta.pdf(grid, alpha_post, beta_post), label="Posterior Beta(16, 5)")
ax.set(xlabel="Вероятность орла p", ylabel="Плотность",
       title="Обновление вероятности орла после 19 бросков")
ax.legend()
plt.show()


## 4. Условные вероятности в Bike Sharing

Порог высокого спроса определите как 75-й процентиль `cnt` **только
на 2011 году**. На 2012 году оцените события $H$ — высокий спрос и
$W$ — `weathersit >= 3`, сравните совместную вероятность с
независимым приближением и обновите две Beta-модели для $P(H|W)$ и
$P(H|\bar W)$. Это хронологическое разделение не использует данные
2012 года для настройки порога. Неопределённость оценённого на
2011 году порога намеренно не моделируется; связь событий в
наблюдательных данных не означает причинного влияния погоды.


### Ответ: условные вероятности и Beta-постериоры Bike Sharing

Единица наблюдения — час. Порог события $H$ «высокий спрос» равен
75-му процентилю `cnt` **только 2011 года**, $q_{0.75}=\mathbf{211}$;
здесь $H=\{cnt\ge211\}$. Событие $W$ — код `weathersit >= 3`, то есть
неблагоприятная погода по определению задания. Вероятности оценены на
8 734 часах 2012 года: $P(H)=\mathbf{0.467254}$,
$P(W)=\mathbf{0.073277}$, $P(H\cap W)=\mathbf{0.017632}$,
$P(H\mid W)=154/640=\mathbf{0.240625}$ и
$P(H\mid\bar W)=3927/8094=\mathbf{0.485174}$.
При независимости ожидалось бы $P(H)P(W)=\mathbf{0.034239}$;
наблюдаемая разница $\mathbf{-0.016607}$ указывает на зависимость
событий в этих наблюдениях.

Для каждой погодной группы с $m$ часами и $s$ часами высокого спроса
prior $\mathrm{Beta}(1,1)$ даёт posterior
$\mathrm{Beta}(1+s,1+m-s)$. Для $W$ это $\mathrm{Beta}(155,487)$:
среднее $\mathbf{0.241433}$, 95%-й credible interval
$\mathbf{[0.209135;\ 0.275256]}$. Для $\bar W$ —
$\mathrm{Beta}(3928,4168)$: среднее $\mathbf{0.485178}$,
интервал $\mathbf{[0.474296;\ 0.496067]}$.
Интервалы условны на выбранном пороге и prior; зависимость соседних часов
сужает их слишком оптимистично. Сезон, время суток и другие факторы
могут смешивать связь погоды и спроса, поэтому причинный эффект здесь
не установлен.


In [ ]:
hour = pd.read_csv(DATA_DIR / "hour.csv", parse_dates=["dteday"])
assert hour.shape == (17_379, 17)
assert (hour["dteday"].dt.year == hour["yr"] + 2011).all()
threshold = hour.loc[hour["yr"] == 0, "cnt"].quantile(0.75)
hour_2012 = hour.loc[hour["yr"] == 1].copy()
hour_2012["high_demand"] = hour_2012["cnt"] >= threshold
hour_2012["bad_weather"] = hour_2012["weathersit"] >= 3
assert len(hour_2012) == 8_734
print(f"Порог по 2011 году: cnt ≥ {threshold:g}")
display(pd.crosstab(hour_2012["bad_weather"], hour_2012["high_demand"],
                    rownames=["Неблагоприятная погода W"],
                    colnames=["Высокий спрос H"]))

high = hour_2012["high_demand"]
bad = hour_2012["bad_weather"]
p_high, p_bad = high.mean(), bad.mean()
p_joint = (high & bad).mean()
p_independent = p_high * p_bad
assert bad.sum() > 0 and (~bad).sum() > 0
p_high_given_bad = high[bad].mean()
p_high_given_good = high[~bad].mean()
display(pd.Series({
    "P(H)": p_high,
    "P(W)": p_bad,
    "P(H ∩ W)": p_joint,
    "P(H)P(W)": p_independent,
    "P(H ∩ W) − P(H)P(W)": p_joint - p_independent,
    "P(H | W)": p_high_given_bad,
    "P(H | не W)": p_high_given_good,
}, name="оценка").to_frame())

posteriors = {}
for condition, label in [(True, "Неблагоприятная"), (False, "Остальная")]:
    group = hour_2012.loc[hour_2012["bad_weather"] == condition]
    m, s = len(group), int(group["high_demand"].sum())
    alpha, beta = 1 + s, 1 + m - s
    low, high_bound = stats.beta.ppf([0.025, 0.975], alpha, beta)
    posteriors[label] = (alpha, beta, alpha / (alpha + beta), low, high_bound)
display(pd.DataFrame.from_dict(posteriors, orient="index",
                               columns=["alpha", "beta", "posterior mean", "CI 2.5%", "CI 97.5%"]))

grid = np.linspace(0.15, 0.55, 501)
fig, ax = plt.subplots(figsize=(8, 4))
for label, (alpha, beta, mean, low, high_bound) in posteriors.items():
    ax.plot(grid, stats.beta.pdf(grid, alpha, beta), label=f"{label}: Beta({alpha}, {beta})")
    ax.axvspan(low, high_bound, alpha=0.10)
ax.set(xlabel="P(высокий спрос | погодная группа)", ylabel="Плотность",
       title="Beta-постериоры на часах 2012 года")
ax.legend()
plt.show()


## Итоговая интерпретация

Дайте 5–8 предложений: перечислите допущения биномиальной модели,
объясните отличие точного и имитационного расчётов, различите
частотный доверительный и байесовский достоверный интервалы и не
делайте причинного вывода о погоде по наблюдательным данным.


### Итог вероятностной части

Для $X\sim\mathrm{Binom}(n,p)$ нужны фиксированное число испытаний,
независимость исходов и одно и то же $p$; общие причины неявок могут
нарушить эти предположения. Для 137 бронирований точная вероятность
$P(X\ge50)$ равна 0.822096, а имитация 100 000 рейсов дала 0.822670:
её отличие вызвано случайной ошибкой Монте-Карло, а не другой моделью.
При вместимости 100 максимум в проверенном диапазоне — 151 бронирование,
поскольку риск 0.048869, а для 152 уже 0.060715. Частотный доверительный
интервал описывает процедуру, которая в повторных выборках покрывает
фиксированный параметр с заданной частотой. Байесовский достоверный
интервал относится к posterior распределению параметра при данных и prior.
На часах Bike Sharing плохая погода связана с меньшей долей высокого
спроса, но наблюдательная связь с возможным смешением факторов не
доказывает причинного влияния погоды.
